# Generate numbers for zeroshot and no covariate runs that show that ensembling different models for weight and smoothness classes yield better performances

In [1]:
import pandas as pd
import os, gc, sys, time
sys.path.append("../src/jobs/")  # adjust as needed to import local modules)

from concurrent.futures import ThreadPoolExecutor, as_completed

from m5_dataprep import M5DataPipeline
from m5_evaluator import M5Evaluator

In [ ]:
# Evaluation phase ends at d_1941; we forecast d_1942-d_1969 (28 days)
_CUTOFF_RAW = '2016-05-22'
DATA_PATH     = '/mnt/lab/datasets/M5/jointed_M5.parquet'
CALENDAR_PATH = '/mnt/lab/nmwamsojo/m5_data/calendar.csv'
ACTUALS_PATH  = '/mnt/lab/nmwamsojo/m5_data/sales_test_evaluation.csv'


CUTOFF_DAY  = (
    pd.to_datetime(_CUTOFF_RAW) - pd.Timedelta(days=28)
).strftime('%Y-%m-%d')

DATA_TAG = 'sales_only'  # used in model tags to denote which data prep was used
print(f'Evaluation cutoff : {CUTOFF_DAY}')

# ── Hyperparameter sweep grid ────────────────────────────────────────────
CONTEXT_LENGTHS = [2, 4, 8, 16, 32, 64, 128, 256, 512, 1024]
BATCH_SIZES     = [64]

Evaluation cutoff : 2016-04-24


# Read input data

In [4]:
dataprep_config = {"tag": DATA_TAG,
                   "base_cols": ['id', 'date', 'sales_quantity'],
                   "extra_cols": []
                }
pipeline = M5DataPipeline(config=dataprep_config)
hist_df, hist_df_trimmed, future_df, static_df, weights_scales = pipeline.get_prepared_data(DATA_PATH, CUTOFF_DAY, level=12) #, force_reprepare=True)

print(f"\nhist_df         : {hist_df.shape}  (columns: {list(hist_df.columns)})")
print(f"hist_df_trimmed : {hist_df_trimmed.shape}")
print(f"future_df       : {future_df.shape}  (columns: {list(future_df.columns)})")
print(f"static_df       : {static_df.shape}")
print(f"weights_scales  : {weights_scales.shape}  levels: {sorted(weights_scales['level'].unique())}")

del pipeline
gc.collect()

--- Cache Hit: Data found in /mnt/lab/nmwamsojo/prepared_data/sales_only/level_12/20160424 ---

hist_df         : (58327370, 18)  (columns: ['id', 'date', 'sales_quantity', 'wm_yr_wk', 'wday', 'month', 'year', 'event_name_1', 'event_type_1', 'snap_CA', 'snap_TX', 'snap_WI', 'sell_price', 'item_id', 'dept_id', 'cat_id', 'store_id', 'state_id'])
hist_df_trimmed : (45942500, 18)
future_df       : (853720, 17)  (columns: ['id', 'date', 'wm_yr_wk', 'wday', 'month', 'year', 'event_name_1', 'event_type_1', 'snap_CA', 'snap_TX', 'snap_WI', 'sell_price', 'item_id', 'dept_id', 'cat_id', 'store_id', 'state_id'])
static_df       : (30490, 6)
weights_scales  : (42840, 7)  levels: [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9), np.int64(10), np.int64(11), np.int64(12)]


0

In [5]:
def _wide_to_long(gt_wide: pd.DataFrame, calendar_path: str) -> pd.DataFrame:
    """
    Convert M5 evaluation CSV (wide, d_1942-d_1969 columns) to long format.
    Calendar join maps day codes (d_XXXX) to real calendar dates.
    """
    if 'id' not in gt_wide.columns:
        gt_wide['id'] = gt_wide['item_id'] + '_' + gt_wide['store_id'] + '_evaluation'
    day_cols = [c for c in gt_wide.columns if c.startswith('d_')]
    long = gt_wide.melt(id_vars=['id'], value_vars=day_cols,
                        var_name='d', value_name='sales_quantity')
    cal = pd.read_csv(calendar_path, usecols=['d', 'date'])
    cal['date'] = pd.to_datetime(cal['date'])
    long = long.merge(cal, on='d', how='left').drop(columns=['d'])
    long['id'] = long['id'].str.replace('_evaluation', '', regex=False)
    return long[['id', 'date', 'sales_quantity']]


# Evaluation window actuals (d_1942-d_1969)
_eval_raw      = pd.read_csv(ACTUALS_PATH)
df_actual_eval = _wide_to_long(_eval_raw, CALENDAR_PATH)
del _eval_raw
print(f'Eval actuals : {df_actual_eval.shape} | '
      f"{df_actual_eval['date'].min().date()} -> {df_actual_eval['date'].max().date()}")

# Full parquet actuals — evaluator uses these to compute WRMSSE
df_actual = (
    pd.read_parquet(DATA_PATH, columns=['id', 'date', 'sold'])
    .rename(columns={'sold': 'sales_quantity'})
)
df_actual['id'] = (
    df_actual['id'].astype(str)
    .str.replace('_evaluation', '', regex=False)
    .str.replace('_validation', '', regex=False)
)
print(f'Full actuals : {df_actual.shape}')


Eval actuals : (853720, 3) | 2016-05-23 -> 2016-06-19
Full actuals : (59181090, 3)


# function

In [26]:
def evaluate_metrics(
    fcst_df: pd.DataFrame,
    actual_df: pd.DataFrame,
    evaluator: M5Evaluator,
) -> dict:
    """
    Evaluate forecast against actuals and return flat metrics dict.

    Parameters
    ----------
    fcst_df   : forecast DataFrame [id, date, <target>]
    actual_df : ground-truth DataFrame [id, date, <target>]
    evaluator : pre-built M5Evaluator instance

    Returns
    -------
    dict with WRMSSE, WAPE_L12, RMSE_L12, MAE_L12, level_scores
    """
    metrics = evaluator.evaluate_all(fcst_df, actual_df)
    return {
        "WRMSSE":   metrics["WRMSSE"],
        "WAPE_L12": metrics["WAPE_L12"],
        "RMSE_L12": metrics["RMSE_L12"],
        "MAE_L12":  metrics["MAE_L12"],
        **metrics.get("level_scores", {}),
    }


def load_and_filter_forecasts(
    fcst_paths: list[str | Path],
    selected_ids_per_file: list[list[str] | None],
) -> pd.DataFrame:
    """
    Load one or multiple forecast files, filter each to its assigned series,
    and concatenate into a single ensemble forecast DataFrame.

    Parameters
    ----------
    fcst_paths            : list of parquet file paths, one per source forecast
    selected_ids_per_file : list of id lists, one per path.
                            Pass None for a given entry to keep all series from
                            that file. Must be the same length as fcst_paths.

    Returns
    -------
    Single concatenated DataFrame of filtered forecasts.

    Raises
    ------
    ValueError        if lengths of the two lists differ, or all files are empty
                      after filtering.
    FileNotFoundError if any path does not exist.
    """
    if len(fcst_paths) != len(selected_ids_per_file):
        raise ValueError(
            f"fcst_paths ({len(fcst_paths)}) and selected_ids_per_file "
            f"({len(selected_ids_per_file)}) must have the same length."
        )

    segments = []
    for path, ids in zip(fcst_paths, selected_ids_per_file):
        path = Path(path)
        if not path.exists():
            raise FileNotFoundError(f"Forecast file not found: {path}")

        fcst = pd.read_parquet(path)

        if ids is not None:
            # None means "keep all"; empty list is an explicit caller mistake
            if len(ids) == 0:
                warnings.warn(
                    f"Empty id list passed for {path.name} — skipping file. "
                    "Pass None to keep all series from a file.",
                    stacklevel=2,
                )
                continue
            fcst = fcst[fcst["id"].isin(ids)].copy()
            if fcst.empty:
                warnings.warn(
                    f"No matching ids found in {path.name} — skipping.",
                    stacklevel=2,
                )
                continue

        segments.append(fcst)
        del fcst

    if not segments:
        raise ValueError("All forecast files were empty after filtering.")

    ensemble = pd.concat(segments, ignore_index=True)
    del segments
    gc.collect()
    return ensemble


def run_experiment_scoped(
    exp: dict,
    cutoff_day: str,
    shared_evaluator: M5Evaluator,
    actual_df: pd.DataFrame,
    selected_ids_per_file: list[list[str] | None],
    extra_fcst_paths: list[str | Path] | None = None,
) -> dict | None:
    """
    Load, filter, ensemble and evaluate one experiment configuration.

    Parameters
    ----------
    exp                   : experiment config dict with 'dataprep_tag' and
                            'model_tag'
    cutoff_day            : forecast cutoff date string
    shared_evaluator      : pre-built M5Evaluator instance
    actual_df             : ground-truth DataFrame
    selected_ids_per_file : list of id lists (or None), one entry per forecast
                            file (primary + any extras). None keeps all series
                            from that file.
    extra_fcst_paths      : optional additional forecast files to ensemble with
                            the primary forecast from pipeline. Must have a
                            corresponding entry in selected_ids_per_file.

    Returns
    -------
    Flat metrics dict, or None if the primary forecast file is missing.
    """
    dtag      = exp["dataprep_tag"]
    model_tag = exp["model_tag"]

    pipeline   = M5DataPipeline({"model_tag": model_tag, "tag": dtag})
    fcst_paths = pipeline.get_forecast_paths(cutoff_day, level=12)

    if not os.path.exists(fcst_paths["forecast"]):
        print(f"  [WARN] {model_tag}: primary forecast not found, skipping.", fcst_paths["forecast"])
        return None

    all_paths = [fcst_paths["forecast"]] + (extra_fcst_paths or [])

    # Strict length check — no silent padding
    if len(selected_ids_per_file) != len(all_paths):
        raise ValueError(
            f"{model_tag}: selected_ids_per_file has {len(selected_ids_per_file)} "
            f"entries but there are {len(all_paths)} forecast files. "
            "Provide one entry per file (use None to keep all series)."
        )

    try:
        fcst_df = load_and_filter_forecasts(all_paths, selected_ids_per_file)
    except (FileNotFoundError, ValueError) as e:
        print(f"  [WARN] {model_tag}: {e}")
        return None

    # Filter actuals to the union of explicitly selected ids.
    # Files with None (keep-all) are excluded from the union — their series
    # are already present in actual_df and we don't want to accidentally drop
    # series that appear in keep-all files.
    explicit_ids = {
        id_
        for ids in selected_ids_per_file
        if ids is not None
        for id_ in ids
    }
    scoped_actual = (
        actual_df[actual_df["id"].isin(explicit_ids)].copy()
        if explicit_ids else actual_df.copy()
    )

    scope_size = (
        len(explicit_ids) if explicit_ids
        else fcst_df["id"].nunique()
    )

    flat = {
        "dataprep_tag": dtag,
        "model_tag":    model_tag,
        "cutoff_day":   cutoff_day,
        "scope_size":   scope_size,
        **evaluate_metrics(fcst_df, scoped_actual, shared_evaluator),
    }

    del fcst_df, scoped_actual
    gc.collect()
    return flat

In [13]:
import os
import re
from pathlib import Path
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots


def _slugify(text: str, max_len: int = 200) -> str:
    text = text.lower()
    text = re.sub(r"[^\w\s-]", "", text)
    text = re.sub(r"[\s_-]+", "_", text)
    return text[:max_len].strip("_")


# ── Visual constants ───────────────────────────────────────────────────────────
_C_WRMSSE  = "#2563eb"   # blue
_C_RMSSE   = "#16a34a"   # green
_C_WAPE    = "#dc2626"   # red

_FONT_FAMILY  = "Courier New, monospace"     # ticks — exact CL values read better in mono
_FONT_TITLE   = "Georgia, serif"
_FONT_LEGEND  = "Georgia, serif"

_SZ_TITLE     = 16
_SZ_AXIS_LBL  = 13
_SZ_TICK      = 11
_SZ_LEGEND    = 11

_LW_MAIN      = 2.5     # primary traces
_LW_DOT       = 2.0     # dotted trace
_MK_SIZE      = 8


def plot_chronos_divergence(
    df: pd.DataFrame,
    title: str,
    out_dir: str = "/mnt/lab/nmwamsojo/plots",
    filename: str | None = None,
) -> str:

    # --- Extract context length ---
    if "context_length" not in df.columns:
        df["context_length"] = df["model_tag"].str.extract(r"_cl(\d+)").astype(float)

    df = df.dropna(subset=["context_length", "WRMSSE", "WAPE_L12", "RMSSE_L12"])
    df["context_length"] = df["context_length"].astype(int)

    # --- Aggregate ---
    summary = (
        df.groupby("context_length")[["WRMSSE", "WAPE_L12", "RMSSE_L12"]]
        .mean()
        .sort_index()
        .reset_index()
    )

    # --- Normalize ---
    def normalize(s):
        d = s.max() - s.min()
        return (s - s.min()) / d if d > 0 else s * 0

    summary["WRMSSE_norm"] = normalize(summary["WRMSSE"])
    summary["RMSSE_norm"]  = normalize(summary["RMSSE_L12"])
    summary["WAPE_norm"]   = normalize(summary["WAPE_L12"]) * 100

    # --- X-axis: uniform index positions, real CL values as labels ----------
    # Replaces xaxis_type="log" which produces intermediate ticks (3,6,10…)
    # that collide with the actual power-of-2 context lengths.
    cl_vals = summary["context_length"].tolist()
    x_idx   = list(range(len(cl_vals)))          # 0,1,2 … evenly spaced
    x_labels = [str(v) for v in cl_vals]         # "2","4","8",…"1024"

    # --- Plot ---
    fig = make_subplots(specs=[[{"secondary_y": True}]])

    fig.add_trace(go.Scatter(
        x=x_idx, y=summary["WRMSSE_norm"],
        name="WRMSSE (Aggregate)",
        mode="lines+markers",
        line=dict(color=_C_WRMSSE, width=_LW_MAIN),
        marker=dict(size=_MK_SIZE, symbol="circle",
                    color=_C_WRMSSE, line=dict(width=1.5, color="#ffffff")),
    ), secondary_y=False)

    fig.add_trace(go.Scatter(
        x=x_idx, y=summary["RMSSE_norm"],
        name="RMSSE_L12 (SKU)",
        mode="lines+markers",
        line=dict(color=_C_RMSSE, width=_LW_DOT, dash="dot"),
        marker=dict(size=_MK_SIZE, symbol="diamond",
                    color=_C_RMSSE, line=dict(width=1.5, color="#ffffff")),
    ), secondary_y=False)

    fig.add_trace(go.Scatter(
        x=x_idx, y=summary["WAPE_norm"],
        name="WAPE (Volume)",
        mode="lines+markers",
        line=dict(color=_C_WAPE, width=_LW_MAIN),
        marker=dict(size=_MK_SIZE, symbol="square",
                    color=_C_WAPE, line=dict(width=1.5, color="#ffffff")),
    ), secondary_y=True)

    # --- Shared axis style dicts ---
    _tick_font  = dict(family=_FONT_FAMILY, size=_SZ_TICK,  color="#374151")
    _label_font = dict(family=_FONT_TITLE,  size=_SZ_AXIS_LBL)

    fig.update_layout(
        title=dict(
            text=f"<b>{title}</b>",
            font=dict(family=_FONT_TITLE, size=_SZ_TITLE, color="#111827"),
            x=0.5, xanchor="center",
        ),
        template="plotly_white",
        paper_bgcolor="#ffffff",
        plot_bgcolor="#f9fafb",
        hovermode="x unified",
        margin=dict(l=72, r=72, t=80, b=64),
        legend=dict(
            orientation="h",
            y=1.06, x=0.5, xanchor="center",
            font=dict(family=_FONT_LEGEND, size=_SZ_LEGEND, color="#374151"),
            bgcolor="rgba(255,255,255,0.85)",
            bordercolor="#e5e7eb", borderwidth=1,
        ),
        # x-axis: uniform positions + real labels
        xaxis=dict(
            tickmode="array",
            tickvals=x_idx,
            ticktext=x_labels,
            tickangle=0,                          # keep labels horizontal
            tickfont=_tick_font,
            title=dict(
                text="Context Length (weeks — historical look-back)",
                font=dict(**_label_font, color="#6b7280"),
                standoff=12,
            ),
            showgrid=True,
            gridcolor="#e5e7eb",
            gridwidth=1,
            zeroline=False,
            linecolor="#d1d5db",
            linewidth=1,
            mirror=True,
        ),
    )

    # --- Left y-axis ---
    fig.update_yaxes(
        title_text="WRMSSE / RMSSE  (normalised, lower = better)",
        title_font=dict(**_label_font, color=_C_WRMSSE),
        title_standoff=12,
        tickfont=_tick_font,
        gridcolor="#e5e7eb",
        gridwidth=1,
        linecolor="#d1d5db",
        linewidth=1,
        mirror=True,
        secondary_y=False,
    )

    # --- Right y-axis ---
    fig.update_yaxes(
        title_text="WAPE  (normalised %, lower = better)",
        title_font=dict(**_label_font, color=_C_WAPE),
        title_standoff=12,
        tickfont=_tick_font,
        gridcolor="rgba(0,0,0,0)",   # suppress right-axis grid to avoid double grid
        linecolor="#d1d5db",
        linewidth=1,
        secondary_y=True,
    )

    # --- File handling ---
    Path(out_dir).mkdir(parents=True, exist_ok=True)

    if filename is None:
        filename = _slugify(title) + ".html"

    final_path = os.path.join(out_dir, filename)
    tmp_path   = final_path + ".tmp"

    fig.write_html(tmp_path, include_plotlyjs="cdn")
    os.replace(tmp_path, final_path)

    print(f"[Plot saved] http://localhost:8050/{filename}")

# Metrics for all tags

In [24]:
_t0 = time.perf_counter()
# Trim each series to its first nonzero sale (used for L12 scales + all weights)

print(f'  Raw hist     : {hist_df.shape}')
print(f'  Trimmed hist : {hist_df_trimmed.shape}')

# M5Evaluator will parallelise scale computation across all CPU cores
# and optionally use CuPy on cuda:0 for the inner diff/mean ops.
evaluator = M5Evaluator(
    raw_train_df     = hist_df,
    trimmed_train_df = hist_df_trimmed,
    static_df        = static_df,
    weights_df       = weights_scales,
    target_col       = 'sales_quantity',
    price_col        = 'sell_price',
    use_gpu          = True,   # CuPy on cuda:0; silent noop if CuPy not installed
    n_jobs           = -1,     # all cores minus 2 headroom
)
print(f'Evaluator ready in {time.perf_counter()-_t0:.1f}s')

  Raw hist     : (58327370, 18)
  Trimmed hist : (45942500, 18)
  [CPU] CuPy not available — scale computation on 18 CPU cores.
  Building hierarchy scales and weights …
Evaluator ready in 0.0s


In [ ]:
# Define your limited scope here (e.g., specific high-value SKUs)
MY_SELECTED_IDS = hist_df_trimmed.id.tolist()  # Top 100 by weight as an example

# Leave 2 cores for the OS and VSCode server processes visible in htop
MAX_WORKERS = min(16, os.cpu_count() - 2)
print(f'Parallel workers: {MAX_WORKERS}')

_t0  = time.perf_counter()
rows = []

with ThreadPoolExecutor(max_workers=MAX_WORKERS) as pool:
    future_to_exp = {
        pool.submit(
            run_experiment_scoped, 
            exp, 
            CUTOFF_DAY, 
            evaluator, 
            df_actual, 
            [MY_SELECTED_IDS] # <--- Pass the scope
        ): exp
        for exp in EXPERIMENT_MATRIX
    }

    done = 0
    for fut in as_completed(future_to_exp):
        exp  = future_to_exp[fut]
        done += 1
        try:
            result = fut.result()
        except Exception as exc:
            print(f"  [ERROR] {exp['model_tag']} -> {exc}")
            continue
        if result is None:
            continue
        rows.append(result)
        tag    = exp['model_tag']
        cached = 'cached' if result.get('_cached') else 'computed'
        print(f"  [{done:>3}/{len(EXPERIMENT_MATRIX)}] {tag}  "
              f"WRMSSE={result['WRMSSE']:.4f}  ({cached})")

print(f'\nBatch complete in {time.perf_counter()-_t0:.1f}s  ({len(rows)} results)')

if not rows:
    print('No results — check that forecast parquets exist on disk.')
else:
    level_cols   = [f'RMSSE_L{l}' for l in range(1, 13)]
    summary_cols = ['dataprep_tag', 'model_tag', 'WRMSSE', 'WAPE_L12', 'RMSE_L12', 'MAE_L12'] + level_cols

    df_results_low = (
        pd.DataFrame(rows)
        .drop(columns=['_cached', 'cutoff_day'], errors='ignore')
        .reindex(columns=summary_cols)
        .sort_values('WRMSSE')
        .reset_index(drop=True)
    )

    # Parse sweep dimensions from model tag for downstream filtering
    df_results_low['context_length'] = (
        df_results_low['model_tag'].str.extract(r'_cl(\d+)$').astype(float)
    )
    df_results_low['batch_size'] = (
        df_results_low['model_tag'].str.extract(r'_bs(\d+)_').astype(float)
    )

    display(
        df_results_low[['dataprep_tag', 'model_tag', 'WRMSSE', 'RMSSE_L12', 'WAPE_L12', 'context_length', 'batch_size']]
        .head(10)
        .style
        .format({'WRMSSE': '{:.4f}', 'RMSE_L12': '{:.4f}', 'WAPE_L12': '{:.4f}',
                 'context_length': '{:.0f}', 'batch_size': '{:.0f}'})
        .background_gradient(subset=['WRMSSE'], cmap='RdYlGn_r')
        .set_caption('Top-10 experiments by WRMSSE (lower is better)')
    )

    print(f"\nBest  WRMSSE = {df_results_low['WRMSSE'].min():.4f}  "
          f"({df_results_low.iloc[0]['model_tag']})")
    print(f"Worst WRMSSE = {df_results_low['WRMSSE'].max():.4f}")



Parallel workers: 16
  [WARN] hpo_cl_ens_fitAll_erratic_cl4: primary forecast not found, skipping. /mnt/lab/nmwamsojo/prepared_data/sales_event_1/level_12/20160424/models/hpo_cl_ens_fitAll_erratic_cl4/forecasts.parquet
  [WARN] hpo_cl_ens_fitAll_erratic_cl2: primary forecast not found, skipping. /mnt/lab/nmwamsojo/prepared_data/sales_event_1/level_12/20160424/models/hpo_cl_ens_fitAll_erratic_cl2/forecasts.parquet
  [WARN] hpo_cl_ens_fitAll_erratic_cl8: primary forecast not found, skipping. /mnt/lab/nmwamsojo/prepared_data/sales_event_1/level_12/20160424/models/hpo_cl_ens_fitAll_erratic_cl8/forecasts.parquet
  [WARN] hpo_cl_ens_fitAll_erratic_cl32: primary forecast not found, skipping. /mnt/lab/nmwamsojo/prepared_data/sales_event_1/level_12/20160424/models/hpo_cl_ens_fitAll_erratic_cl32/forecasts.parquet
  [WARN] hpo_cl_ens_fitAll_erratic_cl64: primary forecast not found, skipping. /mnt/lab/nmwamsojo/prepared_data/sales_event_1/level_12/20160424/models/hpo_cl_ens_fitAll_erratic_cl64/for

,dataprep_tag,model_tag,WRMSSE,RMSSE_L12,WAPE_L12,context_length,batch_size
0,sales_only,hpo_cl_ens_fitAll_erratic_cl2,1.2428,1.051074,0.8548,2,nan
1,sales_only,hpo_cl_ens_fitAll_erratic_cl8,1.2676,0.937838,0.7639,8,nan
2,sales_only,hpo_cl_ens_fitAll_erratic_cl4,1.3530,1.017532,0.8361,4,nan
3,sales_only,hpo_cl_ens_fitAll_erratic_cl16,1.5266,0.877725,0.6984,16,nan
4,sales_only,hpo_cl_ens_fitAll_erratic_cl32,1.8214,0.876162,0.6838,32,nan
5,sales_only,hpo_cl_ens_fitAll_erratic_cl64,1.9483,0.876741,0.6822,64,nan
6,sales_only,hpo_cl_ens_fitAll_erratic_cl128,2.1190,0.881283,0.6768,128,nan
7,sales_only,hpo_cl_ens_fitAll_erratic_cl1024,2.1241,0.878861,0.6731,1024,nan
8,sales_only,hpo_cl_ens_fitAll_erratic_cl256,2.1256,0.879634,0.6755,256,nan
9,sales_only,hpo_cl_ens_fitAll_erratic_cl512,2.1422,0.881636,0.6751,512,nan



Best  WRMSSE = 1.2428  (hpo_cl_ens_fitAll_erratic_cl2)
Worst WRMSSE = 2.1422


# Metrics by Ensembles

In [37]:
import pandas as pd

# Define the descriptive labels from lowest weight to highest weight
quantile_labels = ['Low', 'Medium-Low', 'Medium-High', 'High']

# Create the new column
weights_scales['weight_segs'] = pd.qcut(
    weights_scales['weight'], 
    q=4, 
    labels=quantile_labels
)

# Prefer the column from hist_df_trimmed (set by pipeline cache);
# fall back to weights_scales at L12 if the column was never propagated there.

seg_columns = "smoothness_segs" #   

seg_names = ["Smooth", "Erratic", "Intermittent", "Lumpy"]
undefined_in = "Lumpy"

if "weight" in seg_columns:
    seg_names = quantile_labels
    undefined_in = "Low"


if seg_columns in hist_df_trimmed.columns:
    _seg_src = hist_df_trimmed[["id", seg_columns]].drop_duplicates()
else:
    _seg_src = (
        weights_scales[weights_scales["level"] == 12][["id", seg_columns]]
        .dropna(subset=[seg_columns])
    )

seg_id_map = {
    seg: _seg_src[_seg_src[seg_columns] == seg]["id"].tolist()
    for seg in seg_names
}

# Fold "Undefined" into Lumpy
_undefined = _seg_src[_seg_src[seg_columns] == "Undefined"]["id"].tolist()
seg_id_map[undefined_in].extend(_undefined)

del _seg_src, _undefined

total = sum(len(v) for v in seg_id_map.values())
for seg, ids in seg_id_map.items():
    print(f"  {seg:>14}: {len(ids):>6,} series  ({100*len(ids)/total:.1f}%)")
print(f"  {'TOTAL':>14}: {total:>6,}")

          Smooth:  8,295 series  (27.2%)
         Erratic:  9,999 series  (32.8%)
    Intermittent:  9,999 series  (32.8%)
           Lumpy:  2,197 series  (7.2%)
           TOTAL: 30,490


In [31]:
def evaluate_from_experiment_matrix(
    experiment_matrix: list[dict],
    evaluator: M5Evaluator,
    actual_df: pd.DataFrame,
    cutoff_day: str,
    label: str = "",
) -> dict:
    """
    Assemble ensemble forecast from an experiment matrix and evaluate.

    Each entry in experiment_matrix is expected to have:
        - 'dataprep_tag' : data tag for pipeline path resolution
        - 'model_tag'    : experiment tag (used to locate the forecast file)
        - 'ids'          : list of series ids to extract from that file

    Parameters
    ----------
    experiment_matrix : list of dicts, one per segment
    evaluator         : pre-built M5Evaluator
    actual_df         : ground-truth DataFrame
    cutoff_day        : e.g. "20160424"
    label             : optional name for this run (shown in output)

    Returns
    -------
    flat metrics dict
    """
    fcst_paths   = []
    selected_ids = []

    for entry in experiment_matrix:
        pipeline = M5DataPipeline({
            "model_tag": entry["model_tag"],
            "tag":       entry["dataprep_tag"],
        })
        paths = pipeline.get_forecast_paths(cutoff_day, level=12)

        if not Path(paths["forecast"]).exists():
            raise FileNotFoundError(
                f"Forecast not found for model_tag='{entry['model_tag']}': "
                f"{paths['forecast']}"
            )

        fcst_paths.append(paths["forecast"])
        selected_ids.append(list(entry["ids"]))

    ensemble = load_and_filter_forecasts(fcst_paths, selected_ids)
    metrics  = evaluate_metrics(ensemble, actual_df, evaluator)

    del ensemble
    gc.collect()

    return {"label": label, "cutoff_day": cutoff_day, **metrics}

In [ ]:
import pandas as pd

# Define the descriptive labels from lowest weight to highest weight
quantile_labels = ['Low', 'Medium-Low', 'Medium-High', 'High']

# Create the new column
weights_scales['weight_segs'] = pd.qcut(
    weights_scales['weight'], 
    q=4, 
    labels=quantile_labels
)

# Prefer the column from hist_df_trimmed (set by pipeline cache);
# fall back to weights_scales at L12 if the column was never propagated there.

seg_columns = "smoothness_segs" #   

seg_names = ["Smooth", "Erratic", "Intermittent", "Lumpy"]
undefined_in = "Lumpy"

if "weight" in seg_columns:
    seg_names = quantile_labels
    undefined_in = "Low"


if seg_columns in hist_df_trimmed.columns:
    _seg_src = hist_df_trimmed[["id", seg_columns]].drop_duplicates()
else:
    _seg_src = (
        weights_scales[weights_scales["level"] == 12][["id", seg_columns]]
        .dropna(subset=[seg_columns])
    )

seg_id_map = {
    seg: _seg_src[_seg_src[seg_columns] == seg]["id"].tolist()
    for seg in seg_names
}

# Fold "Undefined" into Lumpy
_undefined = _seg_src[_seg_src[seg_columns] == "Undefined"]["id"].tolist()
seg_id_map[undefined_in].extend(_undefined)

del _seg_src, _undefined

total = sum(len(v) for v in seg_id_map.values())
for seg, ids in seg_id_map.items():
    print(f"  {seg:>14}: {len(ids):>6,} series  ({100*len(ids)/total:.1f}%)")
print(f"  {'TOTAL':>14}: {total:>6,}")


cl_per_seg = {
    "Low":  1,
    "Medium-Low": 1,
    "Medium-High": 1,
    "High": 256,
}

# Build matrix — one entry per segment
EXPERIMENT_MATRIX = [
    {
        "dataprep_tag": "sales_only",
        "model_type":   "Chronos2",
        "model_tag":    f"hpo_cl_ens_fitAll_{seg_name.lower()}_cl{cl_per_seg[seg_name]}",
        "ids":          seg_id_map[seg_name],
    }
    for seg_name in seg_names
]

# Evaluate
result = evaluate_from_experiment_matrix(
    experiment_matrix = EXPERIMENT_MATRIX,
    evaluator         = evaluator,
    actual_df         = df_actual,
    cutoff_day        = CUTOFF_DAY,
    label             = "best_hpo_ensemble",
)

print(f"{result['label']} | WRMSSE: {result['WRMSSE']:.4f} | WAPE: {result['WAPE_L12']:.2%}")

best_hpo_ensemble | WRMSSE: 0.9803 | WAPE: 86.06%


In [42]:
cl_per_seg = {
    "Smooth":  256,
    "Erratic": 1,
    "Intermittent": 1,
    "Lumpy": 8,
}

# Build matrix — one entry per segment
EXPERIMENT_MATRIX = [
    {
        "dataprep_tag": "sales_only",
        "model_type":   "Chronos2",
        "model_tag":    f"hpo_cl_ens_fitAll_{seg_name.lower()}_cl{cl_per_seg[seg_name]}",
        "ids":          seg_id_map[seg_name],
    }
    for seg_name in seg_names
]

# Evaluate
result = evaluate_from_experiment_matrix(
    experiment_matrix = EXPERIMENT_MATRIX,
    evaluator         = evaluator,
    actual_df         = df_actual,
    cutoff_day        = CUTOFF_DAY,
    label             = "best_hpo_ensemble",
)

print(f"{result['label']} for {seg_columns} | WRMSSE: {result['WRMSSE']:.4f} | WAPE: {result['WAPE_L12']:.2%}")

best_hpo_ensemble for smoothness_segs | WRMSSE: 1.0527 | WAPE: 85.51%
